# CAD Integrity Lab
## From a cracked cube to defensible geometry checks

An independent research work sample. This notebook separates **combinatorial topology** from **native CAD validity**. It runs locally; there is no SGS call, Gradio server, or simulated inference response.

Install the project into this notebook's interpreter from the project root:

```bash
python -m pip install -e ".[algebra,cad,notebook,dev]"
```

The saved notebook contains executed results. Run all cells to regenerate them. Interactive plots use the Plotly notebook MIME renderer; `examples/offline_preview.html` is a self-contained browser fallback with the same synthetic experiment.


In [1]:
import importlib.metadata
import importlib.util
import tempfile
from pathlib import Path

import numpy as np
from IPython.display import display

from cad_integrity import (
    BRepHomologyStitchAnalyzer,
    FilteredSimplicialComplex,
    PersistentHomologyEngine,
    RepairPipeline,
    RepairPolicy,
    WeldPolicy,
    analyze_mesh,
)
from cad_integrity.fixtures import cracked_cube, cube, disk, pinched_tetrahedra, torus
from cad_integrity.serialization import dumps
from cad_integrity.visualization import polygonal_audit_figure

print("cad-integrity-lab", importlib.metadata.version("cad-integrity-lab"))
print("NumPy", np.__version__)


cad-integrity-lab 0.1.0
NumPy 2.3.5


## 1. Audit a known synthetic defect

A 10 mm cube has a detached top face offset by 0.002 mm. Its face orientation is also reversed. The fixture is explicitly synthetic; it is not evidence about SGS output quality.

The reported Betti numbers describe the **boundary cell complex over F₂**. We show free edges separately because **b₁ does not count leaks**.

In [2]:
original = cracked_cube()
before = BRepHomologyStitchAnalyzer(original).evaluate_stitch_integrity()
print("Before (b0, b1, b2):", before.homology.betti_numbers)
print("Free boundary edges:", len(before.boundary_edge_ids))
print("Combinatorial closed/oriented manifold:", before.is_closed_oriented_2manifold)
assert before.homology.betti_numbers == (2, 0, 0)
assert len(before.boundary_edge_ids) == 8


Before (b0, b1, b2): (2, 0, 0)
Free boundary edges: 8
Combinatorial closed/oriented manifold: False


In [3]:
before_figure = polygonal_audit_figure(
    original, before, title="Before — 8 free edges; 0.002 mm synthetic gap"
)
before_figure.show(renderer="plotly_mimetype")


## 2. Repair only what the policy authorizes

Vertex welding is opt-in. It is followed by straight-edge deduplication and complete-loop orientation correction. Every component is processed; inconsistent orientation constraints cause rejection. The original data is preserved.

The displacement budget below is a **polygonal vertex displacement** bound, not a claim about the motion of arbitrary native CAD surfaces.

In [4]:
events = []
policy = RepairPolicy(weld=WeldPolicy(tolerance=0.005, max_displacement=0.005))
result = RepairPipeline(policy).run(original, on_event=events.append)
assert result.candidate is not None
assert result.report.after is not None
assert result.report.decision == "topology_checks_passed"

for event in events:
    print(f"[{event.stage}] {event.message}")
print("After (b0, b1, b2):", result.report.after.homology.betti_numbers)
print("Free boundary edges:", len(result.report.after.boundary_edge_ids))
print("Maximum vertex displacement [mm]:", result.report.maximum_vertex_displacement)
print("Changes:", result.report.changes)
assert result.report.after.homology.betti_numbers == (1, 0, 1)
assert BRepHomologyStitchAnalyzer(original).evaluate_stitch_integrity() == before


[analyze] Analyzing the input polygonal boundary
[weld] Attempting explicitly authorized boundary-vertex welding
[orient] Solving orientation constraints across every component
[verify] Re-running diagnostics on the candidate
[complete] topology_checks_passed
After (b0, b1, b2): (1, 0, 1)
Free boundary edges: 0
Maximum vertex displacement [mm]: 0.002000000000000668
Changes: ('Merged 4 vertices and 4 straight edges', 'Reversed complete loops on 1 faces')


In [5]:
after_figure = polygonal_audit_figure(
    result.candidate, result.report.after,
    title="After — combinatorial checks passed; not CAD certification",
)
after_figure.show(renderer="plotly_mimetype")


## 3. Counterexamples are part of the demonstration

The disk is open despite b₁=0. The torus is closed despite b₁=2. The pinched pair of tetrahedra has no free edges but fails the vertex-link test. A visual impression or one scalar invariant is not a substitute for the stated acceptance criteria.

In [6]:
reports = {
    "disk": BRepHomologyStitchAnalyzer(disk()).evaluate_stitch_integrity(),
    "cube surface": BRepHomologyStitchAnalyzer(cube()).evaluate_stitch_integrity(),
    "torus surface": analyze_mesh(torus()),
    "pinched tetrahedra": analyze_mesh(pinched_tetrahedra()),
}
for name, report in reports.items():
    print(
        f"{name:20} Betti={report.homology.betti_numbers} "
        f"free_edges={len(report.boundary_edge_ids):2} "
        f"bad_vertices={report.nonmanifold_vertex_ids} "
        f"closed_manifold={report.is_closed_oriented_2manifold}"
    )
assert not reports["disk"].is_closed_oriented_2manifold
assert reports["torus surface"].is_closed_oriented_2manifold
assert reports["torus surface"].homology.betti_numbers == (1, 2, 1)
assert reports["pinched tetrahedra"].nonmanifold_vertex_ids


disk                 Betti=(1, 0, 0) free_edges= 4 bad_vertices=() closed_manifold=False
cube surface         Betti=(1, 0, 1) free_edges= 0 bad_vertices=() closed_manifold=True
torus surface        Betti=(1, 2, 1) free_edges= 0 bad_vertices=() closed_manifold=True
pinched tetrahedra   Betti=(1, 0, 2) free_edges= 0 bad_vertices=(0,) closed_manifold=False


## 4. Actual persistence, rather than Betti snapshots

Three edges complete a triangular loop at time 2. A face fills that loop at time 3. The H₁ interval is **[2, 3)**. The reducer pairs birth and death simplices; it does not infer pairings from a list of Betti counts.

In [7]:
filtration = FilteredSimplicialComplex({
    (0,): 0, (1,): 0, (2,): 0,
    (0, 1): 1, (1, 2): 1, (0, 2): 2,
    (0, 1, 2): 3,
})
bars = PersistentHomologyEngine(filtration).compute_persistence_intervals()
for bar in bars:
    print(f"H{bar.dimension}: [{bar.birth}, {bar.death if bar.death is not None else 'infinity'})")
assert [(bar.birth, bar.death) for bar in bars if bar.dimension == 1] == [(2, 3)]


H0: [0.0, 1.0)
H0: [0.0, 1.0)
H0: [0.0, infinity)
H1: [2.0, 3.0)


## 5. Native CAD: preserve a curved surface through sewing and STEP

This optional cell uses the real Open CASCADE kernel. It independently copies each face of a cylinder, producing unsewn native faces. Native repair sews them into one solid without replacing the cylindrical surface by a polygon mesh.

The temporary STEP is read back and checked before publication. Files live inside a temporary directory and are deleted after this cell. This is a separate path from the polygonal homology experiment.

In [8]:
if importlib.util.find_spec("OCP") is None:
    print("Native cell skipped: install cad-integrity-lab[cad] in this interpreter.")
else:
    from OCP.BRep import BRep_Builder
    from OCP.BRepBuilderAPI import BRepBuilderAPI_Copy
    from OCP.BRepPrimAPI import BRepPrimAPI_MakeCylinder
    from OCP.TopAbs import TopAbs_FACE
    from OCP.TopExp import TopExp_Explorer
    from OCP.TopoDS import TopoDS_Compound
    from cad_integrity.adapters.ocp import audit_shape, export_checked_step, repair_shape

    cylinder = BRepPrimAPI_MakeCylinder(2.0, 5.0).Shape()
    detached_faces = TopoDS_Compound()
    builder = BRep_Builder()
    builder.MakeCompound(detached_faces)
    explorer = TopExp_Explorer(cylinder, TopAbs_FACE)
    while explorer.More():
        builder.Add(detached_faces, BRepBuilderAPI_Copy(explorer.Current(), True, False).Shape())
        explorer.Next()

    native_result = repair_shape(detached_faces)
    print("Before free native edges:", len(native_result.before.free_edge_ids))
    print("After free native edges:", len(native_result.after.free_edge_ids))
    print("Surfaces:", native_result.after.surface_types)
    print("Native homology:", native_result.after.homology, "— not inferred from trimmed faces")
    print("Operations:", native_result.operations)
    assert native_result.after.accepted_under_policy
    assert "GeomAbs_Cylinder" in dict(native_result.after.surface_types)
    with tempfile.TemporaryDirectory(prefix="cad-integrity-notebook-") as folder:
        exported = export_checked_step(native_result.candidate, Path(folder) / "cylinder.step")
        print("Round-trip accepted:", exported.after_roundtrip.accepted_under_policy)
        print("Volume [mm^3]:", exported.after_roundtrip.solid_volumes_mm3)
        print("Output SHA-256:", exported.output_sha256)
        assert exported.after_roundtrip.accepted_under_policy
        assert np.isclose(exported.after_roundtrip.solid_volumes_mm3[0], 20*np.pi)


Before free native edges: 4
After free native edges: 0
Surfaces: (('GeomAbs_Cylinder', 1), ('GeomAbs_Plane', 2))
Native homology: None — not inferred from trimmed faces
Operations: ('ShapeFix_Shape on copied native geometry', 'BRepBuilderAPI_Sewing; nonmanifold mode disabled', 'One closed shell converted to an oriented solid')

*******************************************************************
******        Statistics on Transfer (Write)                 ******

*******************************************************************
******        Transfer Mode = 0  I.E.  As Is       ******
******        Transferring Shape, ShapeType = 2                      ******
** WorkSession : Sending all data
 Step File Name : /tmp/cad-integrity-notebook-0hdb0d4f/.cad-integrity-ta0fqgqv.step(118 ents)  Write  Done
Round-trip accepted: True
Volume [mm^3]: (62.83185307179585,)
Output SHA-256: b4bfd8293d6afd4d619918edc6fd2e3a37ef62b4ff7df286a4bd4cea1ebe0f88


## 6. Evidence, not a success banner

The serialized report includes the input hash, policy scope, before/after checks, actual changes, and units. General CAD reports intentionally omit unsupported Betti numbers. Engineering certification and continuous surface-distance bounds are not supplied by this project.

In [9]:
print(dumps(result.report))


{
  "after": {
    "boundary_edge_ids": [],
    "collapsed_edge_ids": [],
    "duplicate_face_ids": [],
    "edge_count": 12,
    "face_count": 6,
    "geometric_self_intersections_checked": false,
    "homology": {
      "coefficients": "F2",
      "euler_characteristic": 2,
      "groups": [
        {
          "betti": 1,
          "dimension": 0,
          "torsion": []
        },
        {
          "betti": 0,
          "dimension": 1,
          "torsion": []
        },
        {
          "betti": 1,
          "dimension": 2,
          "torsion": []
        }
      ]
    },
    "homology_unavailable_reason": null,
    "inconsistent_orientation_edge_ids": [],
    "invalid_face_ids": [],
    "nonmanifold_edge_ids": [],
    "nonmanifold_vertex_ids": [],
    "unused_edge_ids": [],
    "unused_vertex_ids": [],
    "vertex_count": 8
  },
  "before": {
    "boundary_edge_ids": [
      4,
      5,
      6,
      7,
      9,
      12,
      14,
      15
    ],
    "collapsed_edge_ids": [

## Next integration boundary

Keep these functions in the package, not in Gradio callbacks. The UI should manage request staging, worker isolation, progress, display, and gated downloads.

**Service status checked September 13, 2026:** the [official SGS-1 Space app](https://huggingface.co/spaces/spectral-labs/SGS-1/raw/main/app.py) now says the research demo has ended. This notebook makes no live inference request. Start with local STEP upload rather than an invented endpoint.

Read `docs/REVIEW.md`, `docs/MATHEMATICS.md`, `docs/GRADIO_HANDOFF.md`, and `docs/VALIDATION.md` for the detailed contracts and validation record.
